In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv
from torch_geometric.loader import LinkNeighborLoader
from torch_geometric.nn import GraphSAGE
from sklearn.linear_model import LogisticRegression
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [5]:
# Load the datasets
cora_dataset = Planetoid(root='/tmp/PubMed', name='PubMed')
data = cora_dataset[0]
data = data.to(device, 'x', 'edge_index')

train_loader = LinkNeighborLoader(
    data,
    batch_size=256,
    shuffle=True,
    neg_sampling_ratio=1.0,
    num_neighbors=[10, 10],
)

model = GraphSAGE(
    data.num_node_features,
    hidden_channels=64,
    num_layers=2,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [6]:
def train():
    model.train()

    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        h = model(batch.x, batch.edge_index)
        h_src = h[batch.edge_label_index[0]]
        h_dst = h[batch.edge_label_index[1]]
        pred = (h_src * h_dst).sum(dim=-1)
        loss = F.binary_cross_entropy_with_logits(pred, batch.edge_label)
        loss.backward()
        optimizer.step()
        total_loss += float(loss) * pred.size(0)

    return total_loss / data.num_nodes

In [7]:
@torch.no_grad()
def test():
    model.eval()
    out = model(data.x, data.edge_index).cpu()

    clf = LogisticRegression()
    clf.fit(out[data.train_mask], data.y[data.train_mask])

    val_acc = clf.score(out[data.val_mask], data.y[data.val_mask])
    test_acc = clf.score(out[data.test_mask], data.y[data.test_mask])

    return val_acc, test_acc

In [9]:
# too slow for 200 epochs
for epoch in range(0, 10):
    loss = train()
    acc = test()[1]
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Accuracy: {acc:.4f}')

Epoch: 000, Loss: 3.8059, Accuracy: 0.7390
Epoch: 001, Loss: 3.7777, Accuracy: 0.6870
Epoch: 002, Loss: 3.7603, Accuracy: 0.7220
Epoch: 003, Loss: 3.7365, Accuracy: 0.7100
Epoch: 004, Loss: 3.7345, Accuracy: 0.6660
Epoch: 005, Loss: 3.7239, Accuracy: 0.6980
Epoch: 006, Loss: 3.7139, Accuracy: 0.6950
Epoch: 007, Loss: 3.7107, Accuracy: 0.7070
Epoch: 008, Loss: 3.6922, Accuracy: 0.7220
Epoch: 009, Loss: 3.6872, Accuracy: 0.7200


In [10]:
torch.save(model.state_dict(), 'pubmed_gsage.pt')